# Dear Abby Seed Dataset Preparation

Workflow for sampling Dear Abby questions (2010-2017) and preparing them for LLM response generation and seed export.


## Notebook Overview
- Load and filter 2010-2017 data
- Perform stratified sampling by year (N=150)
- Export sampled questions for LLM generation
- Transform LLM responses to seed format


## 1. Environment Setup


In [13]:
# Standard library imports
from __future__ import annotations
import json
from pathlib import Path

# Third-party imports
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

pd.options.display.max_colwidth = 120
pd.options.display.max_columns = 20


def find_project_root(markers: tuple[str, ...] = ("pyproject.toml", "poetry.lock", ".git")) -> Path:
    """Ascend from CWD until a repository marker is located."""
    start = Path.cwd().resolve()
    for directory in (start, *start.parents):
        if any((directory / marker).exists() for marker in markers):
            return directory
    return start


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data/humanLabel/raw/dearabby_raw_da_qs.csv"
SAMPLED_OUTPUT_PATH = PROJECT_ROOT / "data/humanLabel/raw/dearabby_sampled_questions.jsonl"
SEED_OUTPUT_PATH = PROJECT_ROOT / "data/humanLabel/seeds/dearabby_seed.jsonl"
SEED_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 150
RANDOM_STATE = 42

# Print relative paths for cleaner output
sampled_path_rel = SAMPLED_OUTPUT_PATH.relative_to(PROJECT_ROOT) if SAMPLED_OUTPUT_PATH.is_relative_to(PROJECT_ROOT) else SAMPLED_OUTPUT_PATH
seed_path_rel = SEED_OUTPUT_PATH.relative_to(PROJECT_ROOT) if SEED_OUTPUT_PATH.is_relative_to(PROJECT_ROOT) else SEED_OUTPUT_PATH

print(f"Sample size: {SAMPLE_SIZE}")
print(f"Random state: {RANDOM_STATE}")
print(f"Sampled questions output: {sampled_path_rel}")
print(f"Seed file output: {seed_path_rel}")


Sample size: 150
Random state: 42
Sampled questions output: data/humanLabel/raw/dearabby_sampled_questions.jsonl
Seed file output: data/humanLabel/seeds/dearabby_seed.jsonl


## 2. Data Loading & Filtering


In [14]:
# Load the CSV
df = pd.read_csv(DATA_PATH)

# Convert year to int and filter for 2010-2017
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df_filtered = df[(df['year'] >= 2010) & (df['year'] <= 2017)].copy()

# Remove rows with missing or empty question text
df_filtered = df_filtered[
    df_filtered['question_only'].notna() & 
    (df_filtered['question_only'].str.strip() != '')
].copy()

print(f"Loaded {len(df):,} total rows")
print(f"Filtered to {len(df_filtered):,} rows (2010-2017, non-empty questions)")

# Check year distribution
print("\nYear distribution:")
year_counts = df_filtered['year'].value_counts().sort_index()
display(year_counts.to_frame('count'))


Loaded 20,034 total rows
Filtered to 5,867 rows (2010-2017, non-empty questions)

Year distribution:


,count
year,
2010,764
2011,779
2012,767
2013,790
2014,736
2015,753
2016,764
2017,514


## 3. Stratified Sampling by Year


In [15]:
def stratified_sample_by_year(df: pd.DataFrame, n: int, random_state: int) -> pd.DataFrame:
    """Sample n questions with stratification by year."""
    # Calculate samples per year
    unique_years = sorted(df['year'].dropna().unique())
    n_years = len(unique_years)
    base_samples = n // n_years
    remainder = n % n_years
    
    sampled_dfs = []
    for i, year in enumerate(unique_years):
        year_df = df[df['year'] == year].copy()
        if len(year_df) == 0:
            continue
        
        # Add one extra sample for first 'remainder' years
        n_samples = base_samples + (1 if i < remainder else 0)
        n_samples = min(n_samples, len(year_df))  # Don't sample more than available
        
        if n_samples > 0:
            year_sample = year_df.sample(n=n_samples, random_state=random_state + i)
            sampled_dfs.append(year_sample)
    
    result = pd.concat(sampled_dfs, ignore_index=True)
    return result.sort_values('year').reset_index(drop=True)


# Perform stratified sampling
sample_df = stratified_sample_by_year(df_filtered, SAMPLE_SIZE, RANDOM_STATE)

print(f"Sampled {len(sample_df)} questions")
print("\nSample distribution by year:")
sample_year_counts = sample_df['year'].value_counts().sort_index()
display(sample_year_counts.to_frame('count'))

# Verify we have the right number
assert len(sample_df) == SAMPLE_SIZE, f"Expected {SAMPLE_SIZE} samples, got {len(sample_df)}"


Sampled 150 questions

Sample distribution by year:


,count
year,
2010,19
2011,19
2012,19
2013,19
2014,19
2015,19
2016,18
2017,18


## 4. Export Sampled Questions for LLM Generation


In [16]:
# Add question_id
sample_df['question_id'] = [f"{i+1:03d}" for i in range(len(sample_df))]

# Export to JSONL for LLM generation
with open(SAMPLED_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for _, row in sample_df.iterrows():
        # Handle date formatting - day is a string, month might be float
        date_str = None
        if pd.notna(row['month']) and pd.notna(row['day']):
            try:
                month_int = int(row['month'])
                day_str = str(row['day']).strip()
                # Pad day if it's a single digit
                day_padded = day_str.zfill(2) if day_str.isdigit() else day_str
                date_str = f"{int(row['year'])}-{month_int:02d}-{day_padded}"
            except (ValueError, TypeError):
                date_str = None
        
        record = {
            "question_id": row['question_id'],
            "year": int(row['year']),
            "month": float(row['month']) if pd.notna(row['month']) else None,
            "day": row['day'] if pd.notna(row['day']) else None,
            "date": date_str,
            "question_text": row['question_only'].strip(),
            "title": row['title'] if pd.notna(row['title']) else None,
            "letterId": row['letterId'] if pd.notna(row['letterId']) else None,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

sampled_path_rel = SAMPLED_OUTPUT_PATH.relative_to(PROJECT_ROOT) if SAMPLED_OUTPUT_PATH.is_relative_to(PROJECT_ROOT) else SAMPLED_OUTPUT_PATH
print(f"Exported {len(sample_df)} sampled questions to {sampled_path_rel}")
print("\nSample record:")
with open(SAMPLED_OUTPUT_PATH, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    sample_record = json.loads(first_line)
    print(json.dumps(sample_record, indent=2, ensure_ascii=False))


Exported 150 sampled questions to data/humanLabel/raw/dearabby_sampled_questions.jsonl

Sample record:
{
  "question_id": "001",
  "year": 2010,
  "month": 7.0,
  "day": "12",
  "date": "2010-07-12",
  "question_text": "this may seem like a silly question, but what is the proper thing to do if fruit drops on the floor at the grocery store? -- wondering in columbus, ga.",
  "title": "Aging Parents' Go It Alone Attitude Can Be Dangerous",
  "letterId": 3
}


## 5. Transform LLM Responses to Seed Format

After running the LLM generation script, load the responses and transform to seed format.


In [17]:
# This cell will be run after LLM generation is complete
# Find the most recent run directory
import glob
import os

run_dirs = glob.glob(str(PROJECT_ROOT / "outputs/runs/dearabby_*"))
if run_dirs:
    latest_run_dir = max(run_dirs, key=os.path.getmtime)
    run_jsonl_path = Path(latest_run_dir) / "run.jsonl"
    # Print relative paths
    run_dir_rel = Path(latest_run_dir).relative_to(PROJECT_ROOT) if Path(latest_run_dir).is_relative_to(PROJECT_ROOT) else Path(latest_run_dir)
    run_jsonl_rel = run_jsonl_path.relative_to(PROJECT_ROOT) if run_jsonl_path.is_relative_to(PROJECT_ROOT) else run_jsonl_path
    print(f"Found run directory: {run_dir_rel}")
    print(f"Run JSONL path: {run_jsonl_rel}")
    print(f"File exists: {run_jsonl_path.exists()}")
else:
    print("No run directories found. Run the LLM generation script first.")
    run_jsonl_path = None


No run directories found. Run the LLM generation script first.


In [18]:
def slugify_model(model_id: str) -> str:
    """Convert model_id to a slug for identifier."""
    safe = str(model_id).lower()
    for ch in ' /\\:':
        safe = safe.replace(ch, '-')
    safe = ''.join(ch for ch in safe if ch.isalnum() or ch == '-')
    return safe.strip('-') or 'model'


def build_identifier(question_id: str, model_id: str) -> str:
    """Build identifier for seed record."""
    model_slug = slugify_model(model_id)
    return f"dearabby_{question_id}_{model_slug}"


def build_prompt_title(question_id: str, year: int) -> str:
    """Build prompt title for seed record."""
    return f"Dear Abby Question {question_id} ({year})"


# Load LLM responses from run.jsonl
if run_jsonl_path and run_jsonl_path.exists():
    run_records = []
    with open(run_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                run_records.append(json.loads(line))
    
    print(f"Loaded {len(run_records)} LLM response records")
    
    # Load sampled questions to get metadata
    sampled_questions = {}
    with open(SAMPLED_OUTPUT_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                q = json.loads(line)
                sampled_questions[q['question_id']] = q
    
    # Transform to seed format
    seed_records = []
    for record in run_records:
        # Extract question_id from prompt_id (format: "dearabby_001")
        prompt_id = record.get('prompt_id', '')
        if prompt_id.startswith('dearabby_'):
            question_id = prompt_id.replace('dearabby_', '')
        else:
            continue
        
        # Get question metadata
        question_meta = sampled_questions.get(question_id)
        if not question_meta:
            continue
        
        # Extract question text from messages
        messages = record.get('messages', [])
        question_text = ""
        for msg in messages:
            if msg.get('role') == 'user':
                content = msg.get('content', '')
                # Remove "I need advice. " prefix if present
                if content.startswith('I need advice. '):
                    question_text = content.replace('I need advice. ', '', 1)
                else:
                    question_text = content
                break
        
        if not question_text:
            continue
        
        # Build seed record
        seed_record = {
            "identifier": build_identifier(question_id, record['model_id']),
            "prompt_title": build_prompt_title(question_id, question_meta['year']),
            "prompt_body": question_text,
            "model_response_text": record.get('response_text', ''),
            "model_id": record['model_id'],
            "run_id": "dearabby-v1",
            "metadata": {
                "source": "dear-abby",
                "year": question_meta['year'],
                "original_date": question_meta.get('date'),
                "question_id": question_id,
            }
        }
        seed_records.append(seed_record)
    
    print(f"Created {len(seed_records)} seed records")
    
    # Export to seed file
    with open(SEED_OUTPUT_PATH, 'w', encoding='utf-8') as f:
        for record in seed_records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    
    seed_path_rel = SEED_OUTPUT_PATH.relative_to(PROJECT_ROOT) if SEED_OUTPUT_PATH.is_relative_to(PROJECT_ROOT) else SEED_OUTPUT_PATH
    print(f"Exported seed file to {seed_path_rel}")
    print("\nSample seed record:")
    if seed_records:
        print(json.dumps(seed_records[0], indent=2, ensure_ascii=False))
else:
    print("Run JSONL file not found. Please run the LLM generation script first.")


Run JSONL file not found. Please run the LLM generation script first.
